<a href="https://colab.research.google.com/github/iiiyahiya111/Project/blob/main/Week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ================================================================
# WEEK 4 — Environment-Conditioned FiLM + Disagreement Gate
# Google Colab — Complete Prototype
# ================================================================

# -----------------------------
# 0. Install / imports
# -----------------------------
!pip -q install albumentations

import os
import glob
import zipfile
import shutil
import random
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from google.colab import drive

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("WEEK 4 — FiLM + Disagreement Gate")
print("=" * 70)
print("Device:", DEVICE)


# ================================================================
# 1. Mount Google Drive
# ================================================================

drive.mount("/content/drive")

print("\nGoogle Drive mounted.")


# ================================================================
# 2. Automatically locate the two ZIP files
# ================================================================

ENV_NAME = "Dataset_1_Image_Environment.zip"
SEV_NAME = "Potato_Severity_Dataset_DEDUPED_FINAL.zip"

SEARCH_ROOTS = [
    "/content",
    "/content/drive/MyDrive"
]

def find_file(filename):
    """Search common Colab locations recursively."""
    matches = []

    for root in SEARCH_ROOTS:
        pattern = os.path.join(root, "**", filename)
        matches.extend(glob.glob(pattern, recursive=True))

    # Remove duplicate paths
    matches = list(dict.fromkeys(matches))

    if matches:
        return matches[0]

    return None


ENV_ZIP = find_file(ENV_NAME)
SEV_ZIP = find_file(SEV_NAME)

# ------------------------------------------------
# Fallback: manually upload if not found
# ------------------------------------------------
if ENV_ZIP is None or SEV_ZIP is None:

    print("\nZIP file(s) were not found automatically.")
    print("Please select the two ZIP files when the upload window appears.\n")

    from google.colab import files

    uploaded = files.upload()

    # Search uploaded files
    if ENV_ZIP is None and ENV_NAME in uploaded:
        ENV_ZIP = "/content/" + ENV_NAME

    if SEV_ZIP is None and SEV_NAME in uploaded:
        SEV_ZIP = "/content/" + SEV_NAME


if ENV_ZIP is None:
    raise FileNotFoundError(
        f"\nCould not find {ENV_NAME}.\n"
        "Place it anywhere inside Google Drive or upload it to Colab."
    )

if SEV_ZIP is None:
    raise FileNotFoundError(
        f"\nCould not find {SEV_NAME}.\n"
        "Place it anywhere inside Google Drive or upload it to Colab."
    )

print("\nEnvironment ZIP:")
print(ENV_ZIP)

print("\nSeverity ZIP:")
print(SEV_ZIP)


# ================================================================
# 3. Extract datasets
# ================================================================

WORK_DIR = "/content/week4_data"
ENV_DIR = os.path.join(WORK_DIR, "environment")
SEV_DIR = os.path.join(WORK_DIR, "severity")

os.makedirs(WORK_DIR, exist_ok=True)

def extract_if_needed(zip_path, out_dir):

    os.makedirs(out_dir, exist_ok=True)

    marker = os.path.join(out_dir, ".extracted")

    if not os.path.exists(marker):
        print(f"\nExtracting:")
        print(zip_path)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(out_dir)

        Path(marker).touch()

    else:
        print(f"\nAlready extracted: {out_dir}")


extract_if_needed(ENV_ZIP, ENV_DIR)
extract_if_needed(SEV_ZIP, SEV_DIR)


# ================================================================
# 4. Locate environment CSV files
# ================================================================

def find_exact(root, filename):

    matches = glob.glob(
        os.path.join(root, "**", filename),
        recursive=True
    )

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found inside {root}"
        )

    return matches[0]


ENV_IMAGE_CSV = find_exact(
    ENV_DIR,
    "synthetic_environment_image_level.csv"
)

ENV_DAILY_CSV = find_exact(
    ENV_DIR,
    "synthetic_environment_daily_v2.csv"
)

ENV_HOURLY_CSV = find_exact(
    ENV_DIR,
    "synthetic_environment_hourly.csv"
)

SEVERITY_CSV = find_exact(
    SEV_DIR,
    "potato_severity_FINAL.csv"
)

print("\nLocated files:")
print("Image-level :", ENV_IMAGE_CSV)
print("Daily       :", ENV_DAILY_CSV)
print("Hourly      :", ENV_HOURLY_CSV)
print("Severity    :", SEVERITY_CSV)


# ================================================================
# 5. Load metadata
# ================================================================

env_image = pd.read_csv(ENV_IMAGE_CSV)
env_daily = pd.read_csv(ENV_DAILY_CSV)
env_hourly = pd.read_csv(ENV_HOURLY_CSV)
severity = pd.read_csv(SEVERITY_CSV)

print("\nDataset shapes:")
print("Environment image-level :", env_image.shape)
print("Environment daily       :", env_daily.shape)
print("Environment hourly      :", env_hourly.shape)
print("Severity dataset        :", severity.shape)


# ================================================================
# 6. Keep only accepted severity samples
# ================================================================

severity = severity[
    severity["final_decision"].astype(str).str.upper() == "KEEP"
].copy()

severity["filename"] = severity["filename"].astype(str)

print("\nKept severity samples:", len(severity))

print(
    "\nSplit distribution:"
)
print(
    severity["split"].value_counts()
)


# ================================================================
# 7. Construct image -> one synthetic environment candidate
# ================================================================
#
# Dataset 1 contains:
#   1153 images
#   10 candidates per image
#
# For the first Week 4 prototype we use candidate_id = 0.
#
# Later you can use all 10 candidates as environment augmentation.
# ================================================================

env_image = env_image[
    env_image["candidate_id"] == 0
].copy()

env_image["filename"] = env_image["filename"].astype(str)

print(
    "\nOne-to-one environment records:",
    len(env_image)
)


# ================================================================
# 8. Build 144-hour environmental sequences
# ================================================================
#
# Available hourly data:
#   temperature
#   RH
#   RH >= 90 indicator
#
# Available daily data:
#   rainfall
#   leaf wetness
#   soil moisture
#
# The daily quantities are expanded uniformly across 24 hours:
#
# rainfall_hour = daily rainfall / 24
# leaf_wetness_hour = daily leaf-wetness / 24
#
# This is a representation choice for the Week 4 prototype.
# ================================================================

KEYS = [
    "filename",
    "image_id",
    "split",
    "class_name",
    "candidate_id"
]

# Select required daily features
daily_keep = env_daily[
    [
        "filename",
        "image_id",
        "split",
        "class_name",
        "candidate_id",
        "day",
        "rainfall_mm",
        "leaf_wetness_hours",
        "soil_moisture"
    ]
].copy()

daily_keep["rainfall_hourly_proxy"] = (
    daily_keep["rainfall_mm"] / 24.0
)

daily_keep["leaf_wetness_hourly_proxy"] = (
    daily_keep["leaf_wetness_hours"] / 24.0
)

# Hourly data
hourly_keep = env_hourly[
    [
        "filename",
        "image_id",
        "split",
        "class_name",
        "candidate_id",
        "hour",
        "day",
        "temperature",
        "rh",
        "rh_ge90"
    ]
].copy()

# Merge daily information into hourly records
hourly_full = hourly_keep.merge(
    daily_keep[
        [
            "filename",
            "image_id",
            "split",
            "class_name",
            "candidate_id",
            "day",
            "rainfall_hourly_proxy",
            "leaf_wetness_hourly_proxy",
            "soil_moisture"
        ]
    ],
    on=[
        "filename",
        "image_id",
        "split",
        "class_name",
        "candidate_id",
        "day"
    ],
    how="left"
)

# Use only the first 144 hours
hourly_full = hourly_full[
    hourly_full["hour"] < 144
].copy()

print(
    "\nHourly rows after 144-hour restriction:",
    len(hourly_full)
)


# ================================================================
# 9. Build environment dictionary
# ================================================================

ENV_FEATURES = [
    "temperature",
    "rh",
    "rainfall_hourly_proxy",
    "leaf_wetness_hourly_proxy",
    "soil_moisture",
    "rh_ge90"
]

print("\nEnvironmental features:")
for i, f in enumerate(ENV_FEATURES):
    print(i, "->", f)


env_sequences = {}

for key, group in hourly_full.groupby(
    ["filename", "candidate_id"]
):

    group = group.sort_values("hour")

    if len(group) < 144:
        continue

    seq = group.iloc[:144][ENV_FEATURES].values.astype(
        np.float32
    )

    env_sequences[key] = seq


print(
    "\nConstructed 144-hour sequences:",
    len(env_sequences)
)


# ================================================================
# 10. Normalize environmental variables
# ================================================================

all_env = np.concatenate(
    list(env_sequences.values()),
    axis=0
)

env_mean = np.nanmean(all_env, axis=0)
env_std = np.nanstd(all_env, axis=0)

env_std[env_std < 1e-6] = 1.0

for key in list(env_sequences.keys()):

    seq = env_sequences[key]

    seq = np.nan_to_num(
        seq,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    seq = (seq - env_mean) / env_std

    env_sequences[key] = seq.astype(np.float32)


# ================================================================
# 11. Match severity images with environmental sequences
# ================================================================

records = []

for _, row in severity.iterrows():

    filename = row["filename"]

    key = (filename, 0)

    if key not in env_sequences:
        continue

    records.append(
        {
            "filename": filename,
            "split": row["split"],
            "class_name": row["class_name"],
            "severity_percent": float(
                row["severity_percent"]
            ),
            "image_path_original": row["image_path"],
            "lesion_mask_path_original": row[
                "lesion_mask_path"
            ],
            "env_key": key
        }
    )

dataset_df = pd.DataFrame(records)

print(
    "\nMatched image + environment samples:",
    len(dataset_df)
)

print(
    dataset_df["split"].value_counts()
)


# ================================================================
# 12. Resolve extracted image/mask files
# ================================================================

ALL_FILES = {}

for root, dirs, files in os.walk(SEV_DIR):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            ALL_FILES[file] = os.path.join(
                root,
                file
            )


def resolve_filename(path_text):

    basename = os.path.basename(
        str(path_text)
    )

    if basename in ALL_FILES:
        return ALL_FILES[basename]

    return None


dataset_df["image_path"] = dataset_df[
    "image_path_original"
].apply(resolve_filename)

dataset_df["mask_path"] = dataset_df[
    "lesion_mask_path_original"
].apply(resolve_filename)


dataset_df = dataset_df.dropna(
    subset=["image_path", "mask_path"]
).reset_index(drop=True)


print(
    "\nSamples with actual image + mask files:",
    len(dataset_df)
)


# ================================================================
# 13. Train / validation / test
# ================================================================

train_df = dataset_df[
    dataset_df["split"] == "train"
].reset_index(drop=True)

valid_df = dataset_df[
    dataset_df["split"] == "valid"
].reset_index(drop=True)

test_df = dataset_df[
    dataset_df["split"] == "test"
].reset_index(drop=True)

print("\nFinal matched datasets:")
print("Train:", len(train_df))
print("Valid:", len(valid_df))
print("Test :", len(test_df))


# ================================================================
# 14. Image transforms
# ================================================================

IMG_SIZE = 256

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.2),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


valid_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


# ================================================================
# 15. Dataset class
# ================================================================

class PotatoEnvironmentDataset(Dataset):

    def __init__(
        self,
        dataframe,
        env_sequences,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.env_sequences = env_sequences
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Image
        image = np.array(
            Image.open(
                row["image_path"]
            ).convert("RGB")
        )

        # Lesion mask
        mask = np.array(
            Image.open(
                row["mask_path"]
            ).convert("L")
        )

        # Binary mask
        mask = (
            mask > 0
        ).astype(np.float32)

        # Environmental sequence
        env = self.env_sequences[
            row["env_key"]
        ]

        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        else:

            image = torch.tensor(
                image.transpose(2, 0, 1),
                dtype=torch.float32
            ) / 255.0

            mask = torch.tensor(
                mask,
                dtype=torch.float32
            )

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        env = torch.tensor(
            env,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "environment": env,
            "filename": row["filename"],
            "severity": torch.tensor(
                row["severity_percent"],
                dtype=torch.float32
            )
        }


# ================================================================
# 16. DataLoaders
# ================================================================

BATCH_SIZE = 8

train_ds = PotatoEnvironmentDataset(
    train_df,
    env_sequences,
    train_transform
)

valid_ds = PotatoEnvironmentDataset(
    valid_df,
    env_sequences,
    valid_transform
)

test_ds = PotatoEnvironmentDataset(
    test_df,
    env_sequences,
    valid_transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ================================================================
# 17. FiLM Layer
# ================================================================

class FiLM(nn.Module):

    """
    Feature-wise Linear Modulation

    y = gamma(env) * x + beta(env)
    """

    def __init__(
        self,
        env_dim,
        num_channels
    ):

        super().__init__()

        self.gamma = nn.Linear(
            env_dim,
            num_channels
        )

        self.beta = nn.Linear(
            env_dim,
            num_channels
        )

    def forward(
        self,
        x,
        env_embedding
    ):

        gamma = self.gamma(
            env_embedding
        ).unsqueeze(-1).unsqueeze(-1)

        beta = self.beta(
            env_embedding
        ).unsqueeze(-1).unsqueeze(-1)

        return (
            gamma * x + beta
        )


# ================================================================
# 18. Environment Encoder
# ================================================================

class EnvironmentEncoder(nn.Module):

    def __init__(
        self,
        input_dim=6,
        hidden_dim=128
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.projection = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, x):

        # x:
        # [B, T, 6]

        output, (h, c) = self.lstm(x)

        # Last bidirectional hidden state
        h_forward = h[-2]
        h_backward = h[-1]

        h = torch.cat(
            [h_forward, h_backward],
            dim=1
        )

        return self.projection(h)


# ================================================================
# 19. Basic Encoder
# ================================================================

class ConvBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class ImageEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.b1 = ConvBlock(3, 32)
        self.b2 = ConvBlock(32, 64)
        self.b3 = ConvBlock(64, 128)
        self.b4 = ConvBlock(128, 256)

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        x1 = self.b1(x)
        p1 = self.pool(x1)

        x2 = self.b2(p1)
        p2 = self.pool(x2)

        x3 = self.b3(p2)
        p3 = self.pool(x3)

        x4 = self.b4(p3)
        p4 = self.pool(x4)

        return p4


# ================================================================
# 20. Decoder block
# ================================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = ConvBlock(
            out_channels,
            out_channels
        )

    def forward(self, x):

        x = self.up(x)
        x = self.conv(x)

        return x


# ================================================================
# 21. Complete Week 4 Model
# ================================================================

class Week4FiLMDisagreementModel(
    nn.Module
):

    def __init__(
        self,
        num_env_features=6,
        env_dim=128,
        num_classes=1
    ):

        super().__init__()

        # ---------------------
        # Image branch
        # ---------------------
        self.image_encoder = ImageEncoder()

        # ---------------------
        # Environment branch
        # ---------------------
        self.environment_encoder = (
            EnvironmentEncoder(
                input_dim=num_env_features,
                hidden_dim=env_dim
            )
        )

        # ---------------------
        # Decoder
        # ---------------------
        self.dec4 = DecoderBlock(
            256,
            128
        )

        self.dec3 = DecoderBlock(
            128,
            64
        )

        self.dec2 = DecoderBlock(
            64,
            32
        )

        self.dec1 = DecoderBlock(
            32,
            16
        )

        # ---------------------
        # FiLM
        # ---------------------
        self.film4 = FiLM(
            env_dim,
            128
        )

        self.film3 = FiLM(
            env_dim,
            64
        )

        self.film2 = FiLM(
            env_dim,
            32
        )

        self.film1 = FiLM(
            env_dim,
            16
        )

        # ---------------------
        # Independent heads
        # ---------------------

        # Image-only prediction
        self.image_head = nn.Conv2d(
            16,
            num_classes,
            kernel_size=1
        )

        # Environment-conditioned prediction
        self.conditional_head = nn.Conv2d(
            16,
            num_classes,
            kernel_size=1
        )

        # Fused prediction
        self.fused_head = nn.Sequential(

            nn.Conv2d(
                32,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                16,
                num_classes,
                kernel_size=1
            )
        )

        # Explicit conflict output
        self.conflict_head = nn.Conv2d(
            16,
            num_classes,
            kernel_size=1
        )

        # ---------------------
        # Learned gate component
        # ---------------------

        self.learned_gate = nn.Sequential(

            nn.Linear(3, 32),

            nn.ReLU(),

            nn.Linear(32, 1),

            nn.Sigmoid()
        )


    def decode_without_environment(
        self,
        z
    ):

        x = self.dec4(z)
        x = self.dec3(x)
        x = self.dec2(x)
        x = self.dec1(x)

        return x


    def decode_with_environment(
        self,
        z,
        env_embedding
    ):

        x = self.dec4(z)
        x = self.film4(
            x,
            env_embedding
        )

        x = self.dec3(x)
        x = self.film3(
            x,
            env_embedding
        )

        x = self.dec2(x)
        x = self.film2(
            x,
            env_embedding
        )

        x = self.dec1(x)
        x = self.film1(
            x,
            env_embedding
        )

        return x


    @staticmethod
    def js_divergence(
        p,
        q,
        eps=1e-8
    ):

        """
        Pixel-wise Jensen-Shannon divergence.
        """

        p = p.clamp(
            min=eps,
            max=1 - eps
        )

        q = q.clamp(
            min=eps,
            max=1 - eps
        )

        m = 0.5 * (p + q)

        kl_pm = (
            p * (
                torch.log(p) -
                torch.log(m)
            )
            +
            (1 - p) * (
                torch.log(1 - p) -
                torch.log(1 - m)
            )
        )

        kl_qm = (
            q * (
                torch.log(q) -
                torch.log(m)
            )
            +
            (1 - q) * (
                torch.log(1 - q) -
                torch.log(1 - m)
            )
        )

        js = 0.5 * (
            kl_pm + kl_qm
        )

        return js


    def forward(
        self,
        image,
        environment,
        gate_mode="hybrid"
    ):

        # ------------------------------------------------
        # Image encoder
        # ------------------------------------------------

        z = self.image_encoder(
            image
        )

        # ------------------------------------------------
        # Environment encoder
        # ------------------------------------------------

        env_embedding = (
            self.environment_encoder(
                environment
            )
        )

        # ------------------------------------------------
        # Independent branches
        # ------------------------------------------------

        image_features = (
            self.decode_without_environment(
                z
            )
        )

        env_features = (
            self.decode_with_environment(
                z,
                env_embedding
            )
        )

        image_logits = (
            self.image_head(
                image_features
            )
        )

        env_logits = (
            self.conditional_head(
                env_features
            )
        )

        # ------------------------------------------------
        # Fused prediction
        # ------------------------------------------------

        fused_features = torch.cat(
            [
                image_features,
                env_features
            ],
            dim=1
        )

        fused_logits = (
            self.fused_head(
                fused_features
            )
        )

        # ------------------------------------------------
        # Conflict output
        # ------------------------------------------------

        conflict_logits = (
            self.conflict_head(
                env_features
            )
        )

        # ------------------------------------------------
        # Branch probabilities
        # ------------------------------------------------

        image_prob = torch.sigmoid(
            image_logits
        )

        env_prob = torch.sigmoid(
            env_logits
        )

        # ------------------------------------------------
        # JS disagreement
        # ------------------------------------------------

        js_map = self.js_divergence(
            image_prob,
            env_prob
        )

        js_score = (
            js_map.mean(
                dim=(1, 2, 3)
            )
        )

        # ------------------------------------------------
        # Branch confidence
        # ------------------------------------------------

        image_confidence = torch.abs(
            image_prob - 0.5
        ).mean(
            dim=(1, 2, 3)
        )

        env_confidence = torch.abs(
            env_prob - 0.5
        ).mean(
            dim=(1, 2, 3)
        )

        # ------------------------------------------------
        # Learned gating inputs
        # ------------------------------------------------

        gate_features = torch.stack(
            [
                js_score,
                image_confidence,
                env_confidence
            ],
            dim=1
        )

        learned_gate = (
            self.learned_gate(
                gate_features
            ).squeeze(1)
        )

        # ------------------------------------------------
        # Gate
        # ------------------------------------------------

        if gate_mode == "learned":

            gate = learned_gate

        elif gate_mode == "js":

            # JS threshold.
            # The learned threshold can later be tuned
            # on validation data.

            gate = torch.sigmoid(
                50.0 *
                (
                    js_score - 0.02
                )
            )

        else:

            # Hybrid gate
            js_gate = torch.sigmoid(
                50.0 *
                (
                    js_score - 0.02
                )
            )

            gate = (
                0.5 * js_gate +
                0.5 * learned_gate
            )

        gate_map = gate.view(
            -1,
            1,
            1,
            1
        )

        # ------------------------------------------------
        # Final routed prediction
        #
        # low disagreement:
        #       fused prediction
        #
        # high disagreement:
        #       conflict prediction
        # ------------------------------------------------

        final_logits = (
            (1.0 - gate_map)
            * fused_logits
            +
            gate_map
            * conflict_logits
        )

        return {
            "image_logits": image_logits,
            "environment_logits": env_logits,
            "fused_logits": fused_logits,
            "conflict_logits": conflict_logits,
            "final_logits": final_logits,

            "image_prob": image_prob,
            "environment_prob": env_prob,

            "js_map": js_map,
            "js_score": js_score,

            "learned_gate": learned_gate,
            "gate": gate
        }


# ================================================================
# 22. Instantiate model
# ================================================================

model = Week4FiLMDisagreementModel(
    num_env_features=len(
        ENV_FEATURES
    ),
    env_dim=128
).to(DEVICE)

print("\nModel created.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 23. Smoke test with a real batch
# ================================================================

batch = next(
    iter(train_loader)
)

images = batch["image"].to(
    DEVICE,
    non_blocking=True
)

environments = batch[
    "environment"
].to(
    DEVICE,
    non_blocking=True
)

masks = batch["mask"].to(
    DEVICE,
    non_blocking=True
)

print("\nInput shapes:")
print("Images       :", images.shape)
print("Environment  :", environments.shape)
print("Masks        :", masks.shape)


with torch.no_grad():

    output = model(
        images,
        environments,
        gate_mode="hybrid"
    )


print("\nOutput shapes:")

for key, value in output.items():

    if torch.is_tensor(value):
        print(
            f"{key:20s}: {tuple(value.shape)}"
        )


# ================================================================
# 24. Loss functions
# ================================================================

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss(
    logits,
    targets,
    eps=1e-6
):

    probs = torch.sigmoid(
        logits
    )

    intersection = (
        probs * targets
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probs.sum(dim=(1, 2, 3))
        +
        targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2.0 * intersection + eps
    ) / (
        denominator + eps
    )

    return 1.0 - dice.mean()


def segmentation_loss(
    logits,
    targets
):

    return (
        bce_loss(
            logits,
            targets
        )
        +
        dice_loss(
            logits,
            targets
        )
    )


# ================================================================
# 25. Week 4 multi-branch loss
# ================================================================

def week4_loss(
    outputs,
    targets
):

    image_loss = segmentation_loss(
        outputs["image_logits"],
        targets
    )

    environment_loss = segmentation_loss(
        outputs["environment_logits"],
        targets
    )

    fused_loss = segmentation_loss(
        outputs["fused_logits"],
        targets
    )

    conflict_loss = segmentation_loss(
        outputs["conflict_logits"],
        targets
    )

    final_loss = segmentation_loss(
        outputs["final_logits"],
        targets
    )

    # Small disagreement regularization.
    #
    # This does not force branches to be identical.
    # It simply avoids uncontrolled divergence
    # during the first prototype stage.
    disagreement_regularization = (
        outputs["js_score"].mean()
    )

    total_loss = (
        1.0 * final_loss
        +
        0.5 * fused_loss
        +
        0.25 * image_loss
        +
        0.25 * environment_loss
        +
        0.10 * conflict_loss
        +
        0.01 * disagreement_regularization
    )

    return total_loss


# ================================================================
# 26. Optimizer
# ================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ================================================================
# 27. One training epoch
# ================================================================
#
# Week 4 goal is architectural implementation.
# This is a smoke-training stage, not final tuning.
# ================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    max_batches=None
):

    model.train()

    total_loss = 0.0
    count = 0

    for batch_idx, batch in enumerate(loader):

        if (
            max_batches is not None
            and batch_idx >= max_batches
        ):
            break

        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        env = batch["environment"].to(
            DEVICE,
            non_blocking=True
        )

        masks = batch["mask"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(
            images,
            env,
            gate_mode="hybrid"
        )

        loss = week4_loss(
            outputs,
            masks
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        total_loss += (
            loss.item()
        )

        count += 1

        if batch_idx % 10 == 0:

            print(
                f"Batch {batch_idx:03d} | "
                f"Loss {loss.item():.4f} | "
                f"Mean JS "
                f"{outputs['js_score'].mean().item():.4f} | "
                f"Gate "
                f"{outputs['gate'].mean().item():.4f}"
            )

    return total_loss / max(
        count,
        1
    )


# ================================================================
# 28. Validation
# ================================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    max_batches=None
):

    model.eval()

    losses = []
    js_scores = []
    gates = []

    for batch_idx, batch in enumerate(loader):

        if (
            max_batches is not None
            and batch_idx >= max_batches
        ):
            break

        images = batch["image"].to(
            DEVICE
        )

        env = batch["environment"].to(
            DEVICE
        )

        masks = batch["mask"].to(
            DEVICE
        )

        outputs = model(
            images,
            env,
            gate_mode="hybrid"
        )

        loss = week4_loss(
            outputs,
            masks
        )

        losses.append(
            loss.item()
        )

        js_scores.extend(
            outputs[
                "js_score"
            ].detach().cpu().numpy()
        )

        gates.extend(
            outputs[
                "gate"
            ].detach().cpu().numpy()
        )

    return {
        "loss": float(
            np.mean(losses)
        ),
        "mean_js": float(
            np.mean(js_scores)
        ),
        "mean_gate": float(
            np.mean(gates)
        )
    }


# ================================================================
# 29. Smoke training
# ================================================================
#
# Run only a few batches first.
# This verifies:
#   - FiLM
#   - environmental encoder
#   - segmentation heads
#   - JS divergence
#   - gate
#   - backpropagation
# ================================================================

print("\n" + "=" * 70)
print("SMOKE TRAINING")
print("=" * 70)

train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    max_batches=20
)

val_stats = evaluate(
    model,
    valid_loader,
    max_batches=5
)

print(
    "\nSmoke training finished."
)

print(
    "Train loss:",
    round(train_loss, 4)
)

print(
    "Validation loss:",
    round(val_stats["loss"], 4)
)

print(
    "Validation mean JS:",
    round(val_stats["mean_js"], 4)
)

print(
    "Validation mean gate:",
    round(val_stats["mean_gate"], 4)
)


# ================================================================
# 30. UNIT TEST 1 — Perfect agreement
# ================================================================

print("\n" + "=" * 70)
print("UNIT TEST 1 — AGREEMENT")
print("=" * 70)

B = 4
H = 32
W = 32

logits_a = torch.zeros(
    B, 1, H, W,
    device=DEVICE
)

logits_b = torch.zeros(
    B, 1, H, W,
    device=DEVICE
)

prob_a = torch.sigmoid(
    logits_a
)

prob_b = torch.sigmoid(
    logits_b
)

js_agree = model.js_divergence(
    prob_a,
    prob_b
).mean(
    dim=(1, 2, 3)
)

print(
    "Mean JS divergence:",
    js_agree.mean().item()
)

assert (
    js_agree.mean().item()
    < 1e-6
), "Agreement test failed."

print(
    "PASS — identical branches produce near-zero JS divergence."
)


# ================================================================
# 31. UNIT TEST 2 — Strong conflict
# ================================================================

print("\n" + "=" * 70)
print("UNIT TEST 2 — STRONG CONFLICT")
print("=" * 70)

logits_image = (
    torch.ones(
        B, 1, H, W,
        device=DEVICE
    ) * 8.0
)

logits_environment = (
    torch.ones(
        B, 1, H, W,
        device=DEVICE
    ) * -8.0
)

prob_image = torch.sigmoid(
    logits_image
)

prob_environment = torch.sigmoid(
    logits_environment
)

js_conflict = model.js_divergence(
    prob_image,
    prob_environment
).mean(
    dim=(1, 2, 3)
)

js_gate = torch.sigmoid(
    50.0 *
    (
        js_conflict - 0.02
    )
)

print(
    "Mean JS divergence:",
    js_conflict.mean().item()
)

print(
    "JS-based gate:",
    js_gate.mean().item()
)

assert (
    js_conflict.mean().item()
    > js_agree.mean().item()
), "Conflict should have larger JS divergence."

assert (
    js_gate.mean().item()
    > 0.5
), "Conflict gate should activate."

print(
    "PASS — strongly conflicting branches activate the gate."
)


# ================================================================
# 32. UNIT TEST 3 — Mixed ambiguous/conflict case
# ================================================================

print("\n" + "=" * 70)
print("UNIT TEST 3 — PARTIAL CONFLICT")
print("=" * 70)

logits_a = torch.zeros(
    B, 1, H, W,
    device=DEVICE
)

logits_b = torch.zeros(
    B, 1, H, W,
    device=DEVICE
)

# First half agrees
logits_a[:, :, :, :W//2] = 5.0
logits_b[:, :, :, :W//2] = 5.0

# Second half conflicts
logits_a[:, :, :, W//2:] = 7.0
logits_b[:, :, :, W//2:] = -7.0

prob_a = torch.sigmoid(
    logits_a
)

prob_b = torch.sigmoid(
    logits_b
)

js_partial_map = model.js_divergence(
    prob_a,
    prob_b
)

js_partial_score = (
    js_partial_map.mean(
        dim=(1, 2, 3)
    )
)

print(
    "Partial-conflict JS:",
    js_partial_score.mean().item()
)

print(
    "Agreement region JS:",
    js_partial_map[
        :, :, :, :W//2
    ].mean().item()
)

print(
    "Conflict region JS:",
    js_partial_map[
        :, :, :, W//2:
    ].mean().item()
)

assert (
    js_partial_map[
        :, :, :, W//2:
    ].mean()
    >
    js_partial_map[
        :, :, :, :W//2
    ].mean()
), "Conflict region should have larger JS."

print(
    "PASS — JS map correctly localizes branch disagreement."
)


# ================================================================
# 33. REAL-BATCH GATING CHECK
# ================================================================

print("\n" + "=" * 70)
print("REAL DATA GATING CHECK")
print("=" * 70)

model.eval()

with torch.no_grad():

    batch = next(
        iter(valid_loader)
    )

    images = batch["image"].to(
        DEVICE
    )

    env = batch["environment"].to(
        DEVICE
    )

    outputs = model(
        images,
        env,
        gate_mode="hybrid"
    )

print(
    "Mean image probability :",
    outputs[
        "image_prob"
    ].mean().item()
)

print(
    "Mean environment probability:",
    outputs[
        "environment_prob"
    ].mean().item()
)

print(
    "Mean JS score:",
    outputs[
        "js_score"
    ].mean().item()
)

print(
    "Mean gate:",
    outputs[
        "gate"
    ].mean().item()
)

print(
    "Minimum gate:",
    outputs[
        "gate"
    ].min().item()
)

print(
    "Maximum gate:",
    outputs[
        "gate"
    ].max().item()
)


# ================================================================
# 34. Save Week 4 model
# ================================================================

SAVE_DIR = "/content/week4_outputs"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    SAVE_DIR,
    "week4_film_disagreement_model.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "env_mean":
            env_mean,

        "env_std":
            env_std,

        "env_features":
            ENV_FEATURES,

        "image_size":
            IMG_SIZE,

        "architecture":
            "CNN + BiLSTM Environment Encoder + FiLM Decoder + JS Disagreement Gate",

        "gate_mode":
            "hybrid"
    },
    MODEL_PATH
)

print(
    "\nModel saved to:"
)

print(
    MODEL_PATH
)


# ================================================================
# 35. Final Week 4 summary
# ================================================================

print("\n" + "=" * 70)
print("WEEK 4 IMPLEMENTATION SUMMARY")
print("=" * 70)

print("""
[✓] Image dataset loaded
[✓] Lesion masks loaded
[✓] Synthetic environmental data loaded
[✓] 144-hour environment sequences constructed
[✓] Environment encoder implemented
[✓] FiLM conditioning implemented
[✓] Independent image branch implemented
[✓] Independent environment-conditioned branch implemented
[✓] Fused segmentation head implemented
[✓] JS-divergence disagreement map implemented
[✓] Learned gate implemented
[✓] Hybrid JS + learned gate implemented
[✓] Conflict output implemented
[✓] Routed final prediction implemented
[✓] Agreement unit test passed
[✓] Strong conflict unit test passed
[✓] Partial conflict localization test passed
[✓] Real-data forward pass completed
[✓] Backpropagation smoke test completed
[✓] Model checkpoint saved

Week 4 core architecture is operational.
""")

print("=" * 70)

WEEK 4 — FiLM + Disagreement Gate
Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted.

Environment ZIP:
/content/drive/MyDrive/Dataset_1_Image_Environment.zip

Severity ZIP:
/content/drive/MyDrive/Potato_Severity_Dataset_DEDUPED_FINAL.zip

Extracting:
/content/drive/MyDrive/Dataset_1_Image_Environment.zip

Extracting:
/content/drive/MyDrive/Potato_Severity_Dataset_DEDUPED_FINAL.zip

Located files:
Image-level : /content/week4_data/environment/synthetic_environment_image_level.csv
Daily       : /content/week4_data/environment/synthetic_environment_daily_v2.csv
Hourly      : /content/week4_data/environment/synthetic_environment_hourly.csv
Severity    : /content/week4_data/severity/Potato_Severity_Dataset_FINAL/severity_dataset/potato_severity_FINAL.csv

Dataset shapes:
Environment image-level : (11530, 24)
Environment daily       : (115300, 25)
Environment hourly      : (27672